In [4]:
from collections import Counter
from pathlib import Path
import pickle, re
import automated_llm_probes as alp

TARGET_N = 700
LOCK20 = [
    "claude-haiku-4.5", "claude-opus-4.5", "claude-opus-4.7", "claude-opus-5",
    "claude-sonnet-4.5", "gpt-3.5-turbo", "gpt-4-turbo", "gpt-4o", "gpt-4o-mini",
    "gpt-5.4", "gpt-5.6-sol", "grok-4.2", "grok-4.3", "grok-4.5", "grok-4.6",
    "grok-build-0.1", "llama-3.1-8b", "llama-3.2-3b", "llama-4-maverick", "llama-4-scout"]
HUMAN_N = {
    "stamp letter send": 558, "superpower": 261, 
    "2305": 101, "execution": 101,
    "belief faith sing": 153, "gloom payment exist": 153, "organ empire comply": 153,
    "petrol diesel pump": 153, "statement stealth detect": 153, "year week embark": 153,
    "frame": 147, "glow": 141, "death": 86, "delay": 86, "enemy": 86,
    "illness": 86, "lie": 86, "marriage": 86, "joy": 85, "shade": 85,
    "simplicity": 85, "sky": 85,}

def targets(n, human_n):
    tot = sum(human_n.values())
    raw = {c: n * k / tot for c, k in human_n.items()}
    out = {c: int(v) for c, v in raw.items()}
    for c in sorted(raw, key=lambda c: raw[c] - out[c], reverse=True):
        if sum(out.values()) >= n:
            break
        out[c] += 1
    return out

def slug(name):
    return re.sub(r"[^\w\-.]+", "-", str(name).strip()).strip("-").lower()

def model_dir(task, name):
    s = slug(name)
    for root in (Path("data") / task / s, Path(task) / s):
        if root.exists():
            return root
    return Path("data") / task / s

def cue_of(row):
    cue = (row.get("kwargs") or {}).get("cue")
    if cue is None:
        cue = row.get("cue")
    if isinstance(cue, (list, tuple)):
        cue = " ".join(str(x) for x in cue if str(x).strip() and str(x).lower() != "nan")
    if not cue:
        parts = [row.get(k) for k in ("cue_0", "cue_1", "cue_2")]
        cue = " ".join(str(x) for x in parts if x and str(x).strip().lower() not in ("", "nan"))
    cue = " ".join(str(cue or "").replace(",", " ").split()).lower()
    if not cue:
        m = re.search(r"words?:\s*(.+)", str(row.get("prompt") or ""), re.I)
        if m:
            cue = " ".join(m.group(1).split("\n")[0].strip(" .").replace(",", " ").split()).lower()
    return cue

def is_title(cue):
    compact = cue.replace(" ", "").replace("(", "").replace(")", "")
    return "2305" in compact or "execution" in compact

def load_row(p):
    try:
        row = pickle.load(open(p, "rb"))
    except Exception:
        return None
    if row.get("error") or not row.get("raw"):
        return None
    return row

tgt = targets(TARGET_N, HUMAN_N)
print("CWT targets", tgt, "sum", sum(tgt.values()))

seen = {}
for m in alp.ready_models():
    if m["name"] in LOCK20 and m["name"] not in seen:
        seen[m["name"]] = m
models = [seen[n] for n in LOCK20 if n in seen]
print("ready", [m["name"] for m in models])
print("not ready", [n for n in LOCK20 if n not in seen])

for m in models:
    have = Counter()
    root = model_dir("cwt", m["name"])
    for p in root.rglob("*.pickle"):
        row = load_row(p)
        if not row:
            continue
        c = cue_of(row)
        if c:
            have[c] += 1
    titled = sum(v for k, v in have.items() if is_title(k))
    title_want = tgt["2305"] + tgt["execution"]
    title_left = max(0, title_want - titled)
    print(f"\n{m['name']}  {sum(have.values())} files  title_pool={titled}/{title_want}  {root}")
    for cue, want in tgt.items():
        if cue in ("2305", "execution"):
            if cue == "2305":
                gap = min(want, title_left)
            else:
                gap = title_left
        else:
            gap = max(0, want - have.get(cue, 0))
        print(f"  {cue:28s} {have.get(cue, 0):4d}/{want:<3d}  {'ok' if gap == 0 else f'+{gap}'}")
        if gap:
            words = ["Execution"] if cue == "execution" else cue.split()
            alp.collect("CWT", models=[m], n_per_model=gap, cue=words, n_to_topup=True)
            have[cue] += gap
            if cue in ("2305", "execution"):
                title_left = max(0, title_left - gap)

CWT targets {'stamp letter send': 127, 'superpower': 59, '2305': 23, 'execution': 23, 'belief faith sing': 35, 'gloom payment exist': 35, 'organ empire comply': 35, 'petrol diesel pump': 35, 'statement stealth detect': 35, 'year week embark': 35, 'frame': 33, 'glow': 32, 'death': 20, 'delay': 20, 'enemy': 20, 'illness': 19, 'lie': 19, 'marriage': 19, 'joy': 19, 'shade': 19, 'simplicity': 19, 'sky': 19} sum 700
ready ['llama-3.1-8b', 'llama-3.2-3b', 'llama-4-maverick', 'llama-4-scout']
not ready []

llama-3.1-8b  720 files  title_pool=30/46  data/cwt/llama-3.1-8b
  stamp letter send              43/127  +84
  llama-3.1-8b: 720 collected, 84 to collect


CWT: 100%|███████████████████████████████████████████████████████████████| 84/84 [05:53<00:00,  4.21s/it]


  superpower                      0/59   +59
  llama-3.1-8b: 804 collected, 59 to collect


CWT: 100%|███████████████████████████████████████████████████████████████| 59/59 [08:11<00:00,  8.34s/it]


  2305                            0/23   +16
  llama-3.1-8b: 863 collected, 16 to collect


...

CWT: 100%|███████████████████████████████████████████████████████████████| 19/19 [02:18<00:00,  7.32s/it]


  simplicity                      0/19   +19
  llama-4-scout: 1098 collected, 19 to collect


CWT: 100%|███████████████████████████████████████████████████████████████| 19/19 [02:15<00:00,  7.13s/it]


  sky                             0/19   +19
  llama-4-scout: 1117 collected, 19 to collect


CWT: 100%|███████████████████████████████████████████████████████████████| 19/19 [02:13<00:00,  7.02s/it]


In [6]:
ready = {m["name"]: m for m in alp.ready_models()}

alp.collect("CWT", models=[ready["grok-4.2"]],
            n_per_model=3, cue=["2305"], n_to_topup=True)
alp.collect("CWT", models=[ready["grok-4.2"]],
            n_per_model=2, cue=["Execution"], n_to_topup=True)

alp.collect("CWT", models=[ready["llama-3.2-3b"]],
            n_per_model=2, cue=["2305"], n_to_topup=True)
alp.collect("CWT", models=[ready["llama-3.2-3b"]],
            n_per_model=1, cue=["Execution"], n_to_topup=True)

  grok-4.2: 1137 collected, 3 to collect


CWT: 100%|█████████████████████████████████████████████████████████████████| 3/3 [00:23<00:00,  7.77s/it]


  grok-4.2: 1140 collected, 2 to collect


CWT: 100%|█████████████████████████████████████████████████████████████████| 2/2 [00:04<00:00,  2.33s/it]


  llama-3.2-3b: 1136 collected, 2 to collect


CWT: 100%|█████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.61s/it]


  llama-3.2-3b: 1138 collected, 1 to collect


CWT: 100%|█████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.63s/it]


In [1]:
import os, pickle
import automated_intelligence_tests as ait
from IPython.display import clear_output

def list_pickle_fps(root):
    fps = []
    for dp, _, fns in os.walk(root):
        for n in fns:
            if n.endswith(".pickle") and not n.endswith(".pickle.tmp"):
                fps.append(os.path.join(dp, n))
    return fps

def dump(p, row):
    tmp = p + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(row, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, p)

def load(p):
    with open(p, "rb") as f:
        return pickle.load(f)

fps = list_pickle_fps("./data/cwt/")
n = len(fps)
for i, p in enumerate(fps, 1):
    row = load(p)
    if isinstance(row.get("score"), (int, float)):
        clear_output(wait=True)
        print(f"{i}/{n}  skip score={row['score']}")
        continue
    raw = row.get("raw")
    try:
        parsed = ait.parse("cwt", raw, stim=row.get("kwargs")) if raw else None
        score = ait.evaluate("cwt", parsed).get("score") if parsed else None
    except Exception:
        parsed, score = None, None
    row["parsed"] = parsed
    row["score"] = score
    dump(p, row)
    raw_show = " ".join(str(raw or "").split())[:120]
    clear_output(wait=True)
    print(f"{i}/{n}  score={score}  raw={raw_show}")

16278/16278  score=0.8064892066536937  raw=In the small town of Willow Creek, whispers of death echoed through the streets. The townspeople were overcome with fear
